# 🚗 拍照估車價：二手車 AI 估價系統（VS Code 本地版）

**拍一張車子照片 → AI 辨識品牌 → 輸入車況 → 預測二手價格**

## 開始之前（只要做一次）
1. VS Code 安裝擴充套件：**Python**、**Jupyter**（左側方塊圖示搜尋安裝）
2. VS Code →「檔案」→「開啟資料夾」→ 選 **`used-car-price` 資料夾**（這個筆記本所在的資料夾）
3. 建立虛擬環境：`Ctrl+Shift+P` → 輸入 **Python: Create Environment** → **Venv** → 選 Python 版本 → 勾選 `requirements.txt`
4. 這個筆記本右上角「**選取核心**」→ 選剛剛建立的 **`.venv`**

然後**從上到下，一格一格按 ▶ 執行**。

| 步驟 | 內容 | 時間（一般筆電 CPU） |
|---|---|---|
| 0 | 檢查環境 | 1 分鐘 |
| 1 | 數值：下載價格資料、訓練價格模型 | 5～15 分鐘 |
| 2 | 圖片：下載車子照片、訓練品牌辨識 | **1～2 小時**（沒有 GPU 比較慢） |
| 3 | 用自己拍的照片測試 | 1 分鐘 |
| 4 | 打開網站 | 1 分鐘 |

> 💡 **圖片訓練太慢的話**：可以只把步驟 2 拿到 Colab（GPU）跑，訓練完把 `models/image` 資料夾下載回來放進這個專案，其他步驟都在本地做。

## 步驟 0：檢查環境

In [ ]:
import os, sys, platform, importlib
print("Python：", sys.version.split()[0], "｜", platform.system())
print("執行位置：", os.getcwd())
print("✅ 在專案資料夾" if os.path.exists("car_config.py") else
      "❌ 目前不在專案資料夾！請用 VS Code「開啟資料夾」打開 used-car-price 資料夾，再重新執行")
missing = [m for m in ["tensorflow", "sklearn", "pandas", "flask", "datasets", "kaggle", "matplotlib", "PIL"]
           if importlib.util.find_spec(m) is None]
if missing:
    print("⚠ 缺少套件：", missing, "→ 請執行下一格安裝")
else:
    import tensorflow as tf
    print("✅ 套件齊全｜TensorFlow", tf.__version__, "｜GPU：", tf.config.list_physical_devices("GPU") or "沒有（用 CPU，正常）")


### 0-1 安裝套件（第一次執行，或上一格說缺少套件時才需要）
裝完後如果還是說找不到，請按上方「**重新啟動**」核心再試一次。

In [ ]:
%pip install -r requirements.txt

---
## 步驟 1：數值部分 — 二手車價格預測
### 1-1 下載價格資料
執行後**VS Code 上方會跳出輸入框**，貼上 Kaggle Token 後按 Enter，接著會問要不要存起來（輸入 `y` 以後就不用再輸入）。

- Token 取得：Kaggle 網站 → 右上角頭像 → Settings → API → **Create New Token**
- 已經下載過會自動略過

In [ ]:
import download_data
download_data.download_price()

### 1-2 訓練價格模型（約 5～15 分鐘）
**你應該會看到：**
```
讀取價格資料：99187 筆 → 清理後 97434 筆，9 個品牌
...
========== 測試集結果 ==========
  基準線(查平均價)   平均誤差 £x,xxx   誤差率 xx.x%   R² 0.xxx
  LinearRegression / RandomForest / Keras神經網路 ...
```
**怎麼判斷：** 三個機器學習模型的誤差率要**低於基準線**；R² 越接近 1 越好。

In [ ]:
!"{sys.executable}" train_price.py

### 1-3 看價格模型的圖表

In [ ]:
from IPython.display import Image, display
import os
for f in ["1_model_compare", "2_pred_vs_actual", "3_feature_importance", "4_nn_training"]:
    path = f"models/price/{f}.png"
    if os.path.exists(path):
        print(f"📊 {f}.png"); display(Image(path, width=600))
    else:
        print(f"⚠ 找不到 {path}，請確認上一格有跑到「✅ 完成」")

---
## 步驟 2：圖片部分 — 品牌辨識
### 2-1 下載車子照片並依品牌分資料夾（約 5～15 分鐘，看網速，不需要 Token）
會從 Hugging Face 下載約 2GB，並把照片整理到 `data/car_images/train`、`data/car_images/test`。**每個品牌都要有照片（不能是 0）**。

In [ ]:
!"{sys.executable}" download_data.py --images

### 2-2 快速測試（約 5～10 分鐘）
只跑 1 個 epoch，**確認程式沒錯誤**，準確率很低是正常的。看到 `✅ 完成！` 再進下一步。

In [ ]:
!"{sys.executable}" train_image.py --quick

### 2-3 正式訓練（一般筆電 CPU 約 1～2 小時）
**你應該會看到：**
- `Epoch 1/12 ... val_accuracy: 0.xx`，**`val_accuracy` 要逐漸上升**
- 微調後會顯示「✔ 微調有效」或「✘ 微調沒有變好，保留第一階段」，兩種都正常
- 最後的「測試集結果」與各品牌的 precision / recall

**怎麼判斷：** 8 類隨便猜約 12.5%，準確率要明顯高於這個數字。

> ⏱ 時間不夠可以少跑幾輪：把下面改成 `train_image.py --epochs 8 --finetune-epochs 10`（準確率會低一點）。
> 過程中**不要按停止**，看到 `✅ 完成！` 才算結束（最後要存模型和畫圖）。

In [ ]:
!"{sys.executable}" train_image.py

### 2-4 看品牌辨識的圖表

In [ ]:
from IPython.display import Image, display
import os
for f in ["1_confusion_matrix", "2_cnn_training", "3_examples"]:
    path = f"models/image/{f}.png"
    if os.path.exists(path):
        print(f"📊 {f}.png"); display(Image(path, width=750))
    else:
        print(f"⚠ 找不到 {path}，請確認上一格有跑到「✅ 完成」")

### 2-5（選做）接著目前的模型繼續訓練
如果 2-3 最後幾輪 `val_accuracy` 還在上升，可以接著練，不用從頭來。
- 會先**備份**目前的模型；**驗證準確率有進步才採用**，沒進步就保留原本的
- 想多學一點：加 `--unfreeze 80`（解凍更多層，但比較容易過擬合）
- 練壞了想還原：`continue_train.py --restore`

In [ ]:
!"{sys.executable}" continue_train.py --epochs 20

---
## 步驟 3：用自己拍的照片測試 ⭐ 很重要
1. 把照片放進專案裡的 **`my_photos`** 資料夾（執行下一格會自動建立）
2. **檔名用實際品牌開頭**，程式會自動算準確率，例如：
   `toyota_1.jpg`、`bmw_停車場.jpg`、`benz_1.jpg`、`vw_1.jpg`、`honda_1.jpg`（不支援的品牌會當作「其他品牌」）、`豐田_2.jpg`
3. 執行下一格

**要拍的品牌：** Toyota、Ford、Hyundai、BMW、Mercedes-Benz、Volkswagen、Audi 各 1～2 台，再加 2～3 台其他品牌（Honda、Nissan…）。
**拍攝技巧：** 車頭或斜前方 45 度、看得到車標、拍到整台車、白天不逆光。

In [ ]:
os.makedirs("my_photos", exist_ok=True)
!"{sys.executable}" test_my_photos.py
from IPython.display import Image, display
if os.path.exists("my_photos_result.png"):
    display(Image("my_photos_result.png"))

---
## 步驟 4：打開網站 🌐
**方法 A（建議）：** VS Code 上方選單「**終端機**」→「新增終端機」，輸入：
```
python app.py
```
看到 `Running on http://127.0.0.1:5000` 後，按住 Ctrl 點那個網址就會打開網站。要關掉網站：在終端機按 `Ctrl+C`。

**方法 B：** 直接執行下一格，會在背景啟動網站並自動打開瀏覽器（關掉網站要按上方「重新啟動」核心）。

In [ ]:
import threading, time, webbrowser, sys
sys.path.insert(0, os.getcwd())
for m in ["car_config", "price_utils", "plot_font", "app"]:
    sys.modules.pop(m, None)
from app import app as flask_app
if not any(t.name == "flask" for t in threading.enumerate()):
    threading.Thread(target=lambda: flask_app.run(port=5000, debug=False, use_reloader=False),
                     name="flask", daemon=True).start()
    time.sleep(3)
webbrowser.open("http://127.0.0.1:5000")
print("✅ 網站已啟動：http://127.0.0.1:5000")